# How to Use the ECCO Sea Ice Velocity Dataset

This notebook demonstrates how to work with the [ECCO L4 Sea Ice Velocity](https://cmr.earthdata.nasa.gov/search/concepts/C1990404790-POCLOUD) dataset using TiTiler-CMR.

## About the Dataset

The ECCO (Estimating the Circulation and Climate of the Ocean) L4 Sea Ice Velocity dataset provides monthly estimates of Arctic sea ice motion on a 0.5-degree latitude/longitude grid. The data spans from 1992 to 2018 and includes two velocity components:
- **SIeice**: Eastward sea ice velocity (m/s)
- **SInice**: Northward sea ice velocity (m/s)

This dataset is valuable for understanding Arctic sea ice dynamics, climate change impacts on polar regions, and ocean circulation patterns.

## What You Will Learn

This notebook covers:
1. Inspecting dataset metadata with the `/compatibility` endpoint
2. Visualizing velocity fields on interactive maps with `/tiles`
3. Computing statistics for custom regions with `/xarray/statistics`
4. Analyzing temporal variations with `/timeseries`
5. Comparing multiple variables (eastward vs. northward components)

In [ ]:
import os
import json

import httpx
from folium import LayerControl, Map, TileLayer


titiler_endpoint = os.getenv(
    "TITILER_CMR_ENDPOINT", "https://openveda.cloud/api/titiler-cmr"
)
collection_concept_id = "C1990404790-POCLOUD"
temporal = "1992-01-16T18:00:00Z"

## Step 1: Dataset Inspection with the Compatibility Endpoint

Before working with any dataset, use the `/compatibility` endpoint to understand:
- Available variables and their data types
- Coordinate dimensions (latitude, longitude, time)
- Temporal coverage and resolution
- Variable value ranges (min, max, mean, percentiles)

This information helps you craft appropriate requests and understand the data structure.

In [ ]:
compatibility_response = httpx.get(
    f"{titiler_endpoint}/compatibility",
    params={"collection_concept_id": collection_concept_id},
    timeout=None,
).json()

print(json.dumps(compatibility_response, indent=2))

The output shows:
- **`SIeice`**: Eastward velocity component (shape: [1, 360, 720])
- **`SInice`**: Northward velocity component (shape: [1, 360, 720])
- **Temporal range**: 1992-01-01 through 2018-01-01 (monthly snapshots)
- **Value ranges**: Both variables range from approximately -0.4 to +0.4 m/s

This tells us the dataset is monthly, global in coverage, and well-suited for tile generation and statistical analysis.

The compatibility metadata shows two variables: `SIeice` and `SInice`, which correspond to the eastward and northward sea ice velocity components.

It also reports a temporal range from `1992-01-01` through `2018-01-01`.

## Step 2: Tile-Based Visualization of Eastward Velocity

Use the `/tiles/WebMercatorQuad` endpoint to render the `SIeice` (eastward) variable as interactive map tiles.
Tiles enable efficient web-based visualization, allowing users to zoom and pan without downloading the entire dataset.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", collection_concept_id),
        ("temporal", temporal),
        ("variables", "SIeice"),
        ("rescale", "-0.4,0.4"),
        ("colormap_name", "rdbu"),
    ),
    timeout=None,
).json()

m = Map(location=[72, -57], zoom_start=3)
TileLayer(
    tiles=r["tiles"][0],
    attr="NASA / TiTiler-CMR",
    name="SIeice eastward velocity",
    overlay=True,
    control=True,
).add_to(m)
LayerControl().add_to(m)
m

## Step 3: Regional Statistics for Eastward Velocity

Use the `/xarray/statistics` endpoint to compute summary statistics for the `SIeice` variable within a custom polygon region.

This is useful for:
- Comparing velocity magnitudes across different regions
- Detecting anomalies or extreme values
- Creating time series of regional averages

## Compute statistics for a region

Use the `/xarray/statistics` endpoint to summarize the `SIeice` variable over a polygon.

In [ ]:
geojson_dict = {
    "type": "Feature",
    "properties": {},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[[-60.0, 70.0], [-60.0, 75.0], [-55.0, 75.0], [-55.0, 70.0], [-60.0, 70.0]]],
    },
}

stats_response = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params={
        "collection_concept_id": collection_concept_id,
        "temporal": temporal,
        "variables": "SIeice",
    },
    json=geojson_dict,
    timeout=None,
).json()

print(json.dumps(stats_response, indent=2))

The statistics output includes detailed summaries:
- `min`, `max`, `mean`: Central tendency and range
- `std`: Standard deviation (variability)
- `median`: 50th percentile
- `histogram`: Distribution of values in the region
- `percentile_2`, `percentile_98`: Extreme percentiles
- `valid_percent`: Coverage percentage in the polygon
- `used_assets`: The granule that was used to compute these statistics

## Step 4: Northward Velocity Component (SInice)

Now let's examine the northward component of sea ice velocity. By comparing both components, we get a complete picture of sea ice motion.

### Tile Visualization of Northward Velocity

Visualize `SInice` (northward velocity) on an interactive map.

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/xarray/WebMercatorQuad/tilejson.json",
    params=(
        ("collection_concept_id", collection_concept_id),
        ("temporal", temporal),
        ("variables", "SInice"),
        ("rescale", "-0.4,0.4"),
        ("colormap_name", "rdbu"),
    ),
    timeout=None,
).json()

m = Map(location=[72, -57], zoom_start=3)
TileLayer(
    tiles=r["tiles"][0],
    attr="NASA / TiTiler-CMR",
    name="SInice northward velocity",
    overlay=True,
    control=True,
).add_to(m)
LayerControl().add_to(m)
m

### Statistics for Northward Velocity

Compute regional statistics for the northward component in the same Arctic region.

In [ ]:
stats_northward = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params={
        "collection_concept_id": collection_concept_id,
        "temporal": temporal,
        "variables": "SInice",
    },
    json=geojson_dict,
    timeout=None,
).json()

print(json.dumps(stats_northward, indent=2))

### Comparison: Eastward vs. Northward

By comparing the statistical outputs from both velocity components, you can assess whether sea ice motion is dominated by meridional (north-south) or zonal (east-west) flow in different regions.

The statistics output is returned for the single band in the selected subset.

To explore the northward component instead, change `variable=SIeice` to `variable=SInice` and rerun the request.

## Time series analysis

Use the `/timeseries` endpoint to get statistics over multiple time steps for the same region.

In [ ]:
timeseries_response = httpx.get(
    f"{titiler_endpoint}/timeseries",
    params={
        "collection_concept_id": collection_concept_id,
        "temporal": "1992-01-01T00:00:00Z/1992-03-01T00:00:00Z",
        "step": "P1M",
        "temporal_mode": "point",
    },
    timeout=None,
).json()

print("Available time steps:")
for item in timeseries_response:
    print(item["temporal"])

In [ ]:
# Get statistics for the first available time step
first_temporal = timeseries_response[0]["temporal"]

timeseries_stats = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params={
        "collection_concept_id": collection_concept_id,
        "temporal": first_temporal,
        "variables": "SIeice",
    },
    json=geojson_dict,
    timeout=None,
).json()

print(f"Statistics for {first_temporal}:")
print(json.dumps(timeseries_stats, indent=2))

## Particle Visualization

This dataset can be visualized as an animated particle flow showing sea ice motion direction and speed.
The particle demo is implemented with [MapLibre GL JS](https://maplibre.org/) and a custom 2D canvas particle system
that decodes the two-band (SIeice, SInice) PNG tiles from TiTiler-CMR to drive particle movement.

See `docs/api/particle_demo/` for the full implementation.

## Next steps

- Use the `/compatibility` endpoint to inspect additional temporal ranges and variable min/max values.
- Change the `temporal` timestamp to view other monthly snapshots in the 1992–2018 series.
You can view the interactive demo at [https://hanzila1.github.io/sea-ice-particle-demo/](https://hanzila1.github.io/sea-ice-particle-demo/).
The source code is available in its [standalone repository](https://github.com/hanzila1/sea-ice-particle-demo).

- Combine both components to compute speed magnitude: `sqrt(SIeice² + SInice²)`.